# NExT-Guard Stage 3: Evaluation (Updated)

Evaluate the streaming safeguard on a held-out test set.
**Update**: Optimized for PyTorch sparse tensor data format.

1. Load intervention config (Threshold & Weights).
2. Simulate streaming inference token-by-token (on Response tokens only).
3. Metrics: F1 Score, APD (Average Precision Delay).

In [ ]:
import os
import yaml
import json
import torch
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from dotenv import load_dotenv
load_dotenv()

from sae_tools.data_loader import get_adapter
from sae_tools.model import filter_data_by_label

In [ ]:
# --- Configuration ---
BASE_DIR = Path.cwd()
MODEL_ROOT = os.getenv("MODEL_ROOT")
DATASET_ROOT = os.getenv("DATASET_ROOT")

INPUT_DIR = BASE_DIR / "results/Guard_Qwen3-8B_20260206_1005"
CONFIG_FILE = INPUT_DIR / "intervention_config.json"
PT_DIR = INPUT_DIR / "predictions"

TEST_DATASET_NAME = "WildGuardTest"
PT_FILE = PT_DIR / f"{TEST_DATASET_NAME}.pt"

print(f"Testing on: {TEST_DATASET_NAME}")

In [ ]:
# 1. Load Config
with open(CONFIG_FILE, 'r') as f:
    config = json.load(f)

THRESHOLD = config['threshold']
feature_map = config['features']
safety_indices = np.array([int(k) for k in feature_map.keys()])
safety_weights = np.array([float(v) for v in feature_map.values()])

# Create Sparse Weight Vector for dot product
num_features_dim = 131072 # Assuming standard SAE size, will confirm from data shape
# Note: We will set shape after loading data to be sure
print(f"Threshold: {THRESHOLD:.4f}, Features: {len(safety_indices)}")

In [ ]:
# 2. Load Test Data & Labels
data_dict = torch.load(PT_FILE, map_location='cpu')
sparse_acts = data_dict['sparse_acts']
valid_token_idx = data_dict['valid_token_idx'].numpy()
seq_lens = data_dict['seq_lens'].numpy()

# Convert to CSR
indices = sparse_acts.indices().numpy()
values = sparse_acts.values().float().numpy()
shape = sparse_acts.shape
X_csr = sp.csr_matrix((values, (indices[0], indices[1])), shape=shape)
num_features_dim = shape[1]

# Load Labels
adapter = get_adapter(TEST_DATASET_NAME)
DATASETS_CONFIG = BASE_DIR / "configs/datasets/datasets_prompt.yaml"
with open(DATASETS_CONFIG, 'r') as f:
    ds_conf = yaml.safe_load(f)
folder = next(d['folder'] for d in ds_conf['datasets'] if d['name'] == TEST_DATASET_NAME)
DATASET_PATH = os.path.join(DATASET_ROOT, folder)

dataset = adapter.load(DATASET_PATH, len(seq_lens))
metadata_list = [item for item in dataset]
LABEL_TYPE = "prompt_label" if "prompt" in str(DATASETS_CONFIG) else "response_label"
y_true = np.array([1 if item.get(LABEL_TYPE) == 'Unsafe' else 0 for item in metadata_list])

In [ ]:
# 3. Streaming Simulation
W_sparse = sp.csr_matrix(
    (safety_weights, (safety_indices, np.zeros_like(safety_indices))),
    shape=(num_features_dim, 1)
)

y_pred = []
intervention_positions = [] # 0.0 to 1.0 relative to VALID sequence length

global_ptr = 0

for i in tqdm(range(len(seq_lens)), desc="Streaming"):
    start_rel, end_rel = valid_token_idx[i]
    abs_start = global_ptr + start_rel
    abs_end = global_ptr + end_rel
    global_ptr += seq_lens[i]
    
    triggered = False
    rel_pos = np.nan
    
    if abs_end > abs_start:
        # Extract valid tokens for this sample
        tokens_csr = X_csr[abs_start:abs_end]
        
        # Compute all scores at once (Batch-Token processing simulates streaming checks)
        scores = tokens_csr.dot(W_sparse).toarray().flatten()
        
        # Find first token that exceeds threshold
        breach_indices = np.where(scores > THRESHOLD)[0]
        
        if len(breach_indices) > 0:
            triggered = True
            first_idx = breach_indices[0]
            valid_len = abs_end - abs_start
            rel_pos = first_idx / valid_len # Normalized position (0=start, 1=end)
            
    y_pred.append(1 if triggered else 0)
    intervention_positions.append(rel_pos)

In [ ]:
# 4. Metrics
print("="*60)
print(classification_report(y_true, y_pred, target_names=['Safe', 'Unsafe']))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Safe', 'Unsafe'])
disp.plot(cmap='Blues')
plt.show()

In [ ]:
# 5. Intervention Timing (True Positives Only)
tp_pos = [p for yt, yp, p in zip(y_true, y_pred, intervention_positions) 
          if yt == 1 and yp == 1]

if tp_pos:
    plt.figure(figsize=(8, 5))
    plt.hist(tp_pos, bins=20, range=(0, 1), color='orange', edgecolor='black', alpha=0.7)
    plt.title("Intervention Position (Relative to Response)")
    plt.xlabel("Position (0.0=Start, 1.0=End)")
    plt.ylabel("Count")
    plt.show()
    print(f"Average Intervention Position: {np.mean(tp_pos):.2%}")
else:
    print("No True Positives to plot.")